# Dataset Preparation — Spam Classification + IMDB Sentiment

This notebook downloads and prepares both datasets used in the project, and saves the
processed train/validation/test CSV files directly to your Google Drive so you don't
need to re-download or re-process them in later notebooks (fine-tuning, TF-IDF baseline, etc).

- **Spam classification:** UCI SMS Spam Collection (~5,574 messages, small and fast)
- **IMDB sentiment:** aclImdb movie reviews, subsampled down from the full 50K reviews
  to keep fine-tuning fast on a T4

Run the cells top to bottom. The first time, Colab will ask for permission to access
your Drive — approve it.

## 1. Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

# All processed datasets will be saved under this folder in your Drive.
# Change this path if you'd rather use a different location.
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/llm_finetune_project"
SPAM_DATA_DIR = os.path.join(DRIVE_PROJECT_DIR, "data", "spam")
IMDB_DATA_DIR = os.path.join(DRIVE_PROJECT_DIR, "data", "imdb")

os.makedirs(SPAM_DATA_DIR, exist_ok=True)
os.makedirs(IMDB_DATA_DIR, exist_ok=True)

print("Spam data will be saved to:", SPAM_DATA_DIR)
print("IMDB data will be saved to:", IMDB_DATA_DIR)

Mounted at /content/drive
Spam data will be saved to: /content/drive/MyDrive/llm_finetune_project/data/spam
IMDB data will be saved to: /content/drive/MyDrive/llm_finetune_project/data/imdb


## 2. Install dependencies

In [2]:
!pip install -q pandas requests

## 3. Spam classification — UCI SMS Spam Collection

Downloads the dataset, balances the classes (spam is only ~13% of the raw data, so we
downsample "ham" to match), and splits into train/validation/test CSVs.

In [3]:
import os
import zipfile
import requests
import pandas as pd


def download_and_extract_spam_dataset(dataset_url, zip_path, extracted_path):
    if os.path.exists(extracted_path) and os.path.exists(os.path.join(extracted_path, "SMSSpamCollection")):
        print(f"`{extracted_path}` already prepared. Skipping download.")
        return

    print("Downloading SMS Spam Collection ...")
    response = requests.get(dataset_url, stream=True, timeout=60)
    response.raise_for_status()

    with open(zip_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)

    print("Extracting ...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extracted_path)


def load_spam_dataframe(extracted_path):
    data_file = os.path.join(extracted_path, "SMSSpamCollection")
    df = pd.read_csv(data_file, sep="\t", header=None, names=["label_str", "text"])
    df["label"] = df["label_str"].map({"ham": 0, "spam": 1})
    df = df.drop(columns=["label_str"])
    df = df.sample(frac=1, random_state=123).reset_index(drop=True)
    return df


def balance_spam_dataframe(df):
    # Spam is the minority class (~13%) - downsample ham to match, so the
    # classifier can't just get high accuracy by predicting "ham" every time.
    n_spam = df[df["label"] == 1].shape[0]
    ham_subset = df[df["label"] == 0].sample(n_spam, random_state=123)
    balanced_df = pd.concat([ham_subset, df[df["label"] == 1]])
    return balanced_df.sample(frac=1, random_state=123).reset_index(drop=True)


def partition_and_save(df, output_dir, train_frac=0.7, val_frac=0.1):
    df_shuffled = df.sample(frac=1, random_state=123).reset_index(drop=True)
    train_end = int(len(df_shuffled) * train_frac)
    val_end = train_end + int(len(df_shuffled) * val_frac)

    train = df_shuffled.iloc[:train_end]
    val = df_shuffled.iloc[train_end:val_end]
    test = df_shuffled.iloc[val_end:]

    train.to_csv(os.path.join(output_dir, "train.csv"), index=False)
    val.to_csv(os.path.join(output_dir, "validation.csv"), index=False)
    test.to_csv(os.path.join(output_dir, "test.csv"), index=False)

    print(f"Saved to {output_dir}")
    print(f"  Train: {len(train)} | Val: {len(val)} | Test: {len(test)}")


spam_dataset_url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
spam_zip_path = "/content/sms_spam_collection.zip"
spam_extracted_path = "/content/sms_spam_collection"

download_and_extract_spam_dataset(spam_dataset_url, spam_zip_path, spam_extracted_path)

spam_df = load_spam_dataframe(spam_extracted_path)
print("\nRaw label distribution:")
print(spam_df["label"].value_counts())

spam_df_balanced = balance_spam_dataframe(spam_df)
print("\nBalanced label distribution:")
print(spam_df_balanced["label"].value_counts())

partition_and_save(spam_df_balanced, SPAM_DATA_DIR)

Extracting ...

Raw label distribution:
label
0    4825
1     747
Name: count, dtype: int64

Balanced label distribution:
label
0    747
1    747
Name: count, dtype: int64
Saved to /content/drive/MyDrive/llm_finetune_project/data/spam
  Train: 1045 | Val: 149 | Test: 300


## 4. IMDB sentiment — subsampled aclImdb

Downloads the full aclImdb dataset, then takes a balanced subset (default: 3,000 train /
500 validation / 1,000 test, evenly split between positive/negative) rather than all 50K
reviews, to keep fine-tuning fast on a T4. Adjust `SUBSET_SIZES` below if you want more or
less data.

In [4]:
import os
import sys
import tarfile
import time
import requests
import pandas as pd


def reporthook(count, block_size, total_size):
    global start_time
    if count == 0:
        start_time = time.time()
    else:
        duration = time.time() - start_time
        progress_size = int(count * block_size)
        percent = count * block_size * 100 / total_size if total_size else 0
        speed = int(progress_size / (1024 * duration)) if duration else 0
        sys.stdout.write(
            f"\r{int(percent)}% | {progress_size / (1024**2):.2f} MB "
            f"| {speed:.2f} MB/s | {duration:.2f} sec elapsed"
        )
        sys.stdout.flush()


def download_and_extract_imdb(dataset_url, target_file, directory):
    if os.path.exists(directory):
        print(f"`{directory}` already exists. Skipping download.")
        return

    if os.path.exists(target_file):
        os.remove(target_file)

    response = requests.get(dataset_url, stream=True, timeout=60)
    response.raise_for_status()

    with open(target_file, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)

    print("\nExtracting dataset ...")
    with tarfile.open(target_file, "r:gz") as tar:
        tar.extractall()


def load_imdb_subset(basepath="aclImdb", per_class_counts=None):
    """
    per_class_counts: dict like {"train_pos": 1500, "train_neg": 1500, "test_pos": ...}
    controls how many files are read per subset/label, instead of loading all 50K.
    """
    data_frames = []
    for subset in ("train", "test"):
        for label_name, label_val in (("pos", 1), ("neg", 0)):
            path = os.path.join(basepath, subset, label_name)
            files = sorted(os.listdir(path))

            key = f"{subset}_{label_name}"
            limit = per_class_counts.get(key) if per_class_counts else None
            if limit is not None:
                files = files[:limit]

            for file in files:
                with open(os.path.join(path, file), "r", encoding="utf-8") as infile:
                    data_frames.append(pd.DataFrame({"text": [infile.read()], "label": [label_val]}))

    df = pd.concat(data_frames, ignore_index=True)
    df = df.sample(frac=1, random_state=123).reset_index(drop=True)
    return df


def partition_and_save_imdb(df, output_dir, sizes):
    """sizes: (train_size, val_size, test_size) - total rows, taken after shuffling."""
    df_shuffled = df.sample(frac=1, random_state=123).reset_index(drop=True)

    train_end = sizes[0]
    val_end = sizes[0] + sizes[1]

    train = df_shuffled.iloc[:train_end]
    val = df_shuffled.iloc[train_end:val_end]
    test = df_shuffled.iloc[val_end:val_end + sizes[2]]

    train.to_csv(os.path.join(output_dir, "train.csv"), index=False)
    val.to_csv(os.path.join(output_dir, "validation.csv"), index=False)
    test.to_csv(os.path.join(output_dir, "test.csv"), index=False)

    print(f"Saved to {output_dir}")
    print(f"  Train: {len(train)} | Val: {len(val)} | Test: {len(test)}")


imdb_dataset_url = "http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
print("Downloading aclImdb (this is a larger download, ~80MB compressed) ...")
download_and_extract_imdb(imdb_dataset_url, "/content/aclImdb_v1.tar.gz", "/content/aclImdb")

# Only read enough raw files per class to comfortably cover the subset sizes below,
# rather than loading and shuffling all 50K reviews into memory.
# NOTE: these must add up to comfortably MORE than sum(SUBSET_SIZES) below,
# since partition_and_save_imdb slices sequentially out of whatever gets loaded here.
# 1500+1500+750+750 = 4500 loaded rows, comfortably above the 4500 total requested
# by SUBSET_SIZES=(3000, 500, 1000), with a small safety margin.
PER_CLASS_READ_LIMIT = {
    "train_pos": 1600, "train_neg": 1600,
    "test_pos": 800, "test_neg": 800,
}

print("\nLoading a subset of aclImdb into a dataframe ...")
imdb_df = load_imdb_subset("/content/aclImdb", per_class_counts=PER_CLASS_READ_LIMIT)
print(f"Loaded {len(imdb_df)} rows total")
print(imdb_df["label"].value_counts())

# Final train/val/test split sizes - deliberately small, matched roughly to the
# spam dataset's scale, to keep fine-tuning time comparable across both tasks.
SUBSET_SIZES = (3000, 500, 1000)
assert sum(PER_CLASS_READ_LIMIT.values()) >= sum(SUBSET_SIZES), (
    f"PER_CLASS_READ_LIMIT only loads {sum(PER_CLASS_READ_LIMIT.values())} rows, "
    f"but SUBSET_SIZES needs {sum(SUBSET_SIZES)}. Increase PER_CLASS_READ_LIMIT."
)

partition_and_save_imdb(imdb_df, IMDB_DATA_DIR, sizes=SUBSET_SIZES)


Extracting dataset ...


/tmp/ipykernel_1058/3559129762.py:43: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall()



Loading a subset of aclImdb into a dataframe ...
Loaded 4800 rows total
label
1    2400
0    2400
Name: count, dtype: int64
Saved to /content/drive/MyDrive/llm_finetune_project/data/imdb
  Train: 3000 | Val: 500 | Test: 1000


## 5. Quick sanity check

Confirms both datasets saved correctly to Drive and previews a few rows of each.

In [5]:
import pandas as pd

print("=== Spam ===")
spam_train = pd.read_csv(f"{SPAM_DATA_DIR}/train.csv")
print(spam_train.shape)
print(spam_train.head(3))

print("\n=== IMDB ===")
imdb_train = pd.read_csv(f"{IMDB_DATA_DIR}/train.csv")
print(imdb_train.shape)
print(imdb_train.head(3))

=== Spam ===
(1045, 2)
                                                text  label
0  Water logging in desert. Geoenvironmental impl...      0
1  Do you want a new Video phone? 600 anytime any...      1
2  Congratulations ur awarded either £500 of CD g...      1

=== IMDB ===
(3000, 2)
                                                text  label
0  I was pulled into this movie early on, much to...      0
1  What can I say after I say the one line summar...      0
2  I was so disappointed in this movie. I don't k...      0
